# Day 4

## 用程式碼做 Token 化

In [ ]:
import tiktoken

encoding = tiktoken.encoding_for_model("gpt-4.1-mini")

tokens = encoding.encode("Hi my name is Ed and I like banoffee pie")

In [ ]:
tokens

In [ ]:
for token_id in tokens:
    token_text = encoding.decode([token_id])
    print(f"{token_id} = {token_text}")

In [ ]:
encoding.decode([326])

# 另一個主題！

### 「記憶」的幻覺

許多人已經知道。但若還不知道——這可能是「啊哈」時刻！

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if not api_key:
    print("No API key was found - please head over to the troubleshooting notebook in this folder to identify & fix!")
elif not api_key.startswith("sk-proj-"):
    print("An API key was found, but it doesn't start sk-proj-; please check you're using the right key - see troubleshooting notebook")
else:
    print("API key found and looks good so far!")

### 你應該很清楚下一個儲存格在做什麼！

_我正在建立 OpenAI Python Client Library 的新實例，這是對呼叫 GPT LLM 或其他 LLM 供應商 endpoint 的 HTTP 請求的輕量包裝_

In [ ]:
from openai import OpenAI

openai = OpenAI()

### 傳給 OpenAI 的 message 是 dict 的清單

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Ed!"}
    ]

In [ ]:
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response.choices[0].message.content

### 好，現在問一個後續問題

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "What's my name?"}
    ]

In [ ]:
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response.choices[0].message.content

### 等等，什麼？？

我們剛告訴你了！

怎麼回事？？

重點是：每次呼叫 LLM 都是完全**無狀態（stateless）**的。每次都是全新的呼叫。作為 AI 工程師，**我們的任務**是設計技巧，讓 LLM 看起來有「記憶」。

In [ ]:
messages = [
    {"role": "system", "content": "You are a helpful assistant"},
    {"role": "user", "content": "Hi! I'm Ed!"},
    {"role": "assistant", "content": "Hi Ed! How can I assist you today?"},
    {"role": "user", "content": "What's my name?"}
    ]

In [ ]:
response = openai.chat.completions.create(model="gpt-4.1-mini", messages=messages)
response.choices[0].message.content

## 總結

若這對你顯而易見，抱歉——但仍值得強調：

1. 每次 LLM 呼叫都是無狀態的
2. 我們每次都在輸入提示詞中傳入到目前為止的完整對話
3. 這造成 LLM 有記憶的幻覺——它似乎記得對話脈絡
4. 但這是技巧；是每次提供完整對話的副產品
5. LLM 只是預測序列中最可能的下一個 token；若序列包含「My name is Ed」之後又有「What's my name?」，它就會預測……Ed！

ChatGPT 產品正是用此技巧——每次你發送訊息，都是整段對話被傳入。

「這表示每次都要為到目前為止的對話多付費嗎」

沒錯。而且這正是我們**想要的**。我們要 LLM 回顧整段對話來預測下一個 token。我們需要這項運算，所以得為電費買單！

